## Prueba técnica<br>Ciencia de Datos

### Marco Israel Rodríguez Cornejo


El objetivo de este notebook es implementar modelos de machine learnig para predecir variables 
usando la base de datos "Default of Credit Card Clients"

In [12]:
# El primer paso es importar la base de datos, este código descarga la base de datos y la carga como un DataFrame.
import pandas as pd
from urllib.request import urlretrieve
from zipfile import ZipFile
import os

if not os.path.isfile("data.zip"):
    url = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"
    urlretrieve(url, "data.zip")

with ZipFile("data.zip", 'r') as zip:
    zip.extractall("data")

data = pd.read_excel("data/default of credit card clients.xls", skiprows=1, index_col=0)
data.keys()

Index(['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2',
       'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default payment next month'],
      dtype='object')

Las variables a predecir son "PAY_AMT4" y "default payment next month".

Las variables con índice 6, 5, 4 corresponden a los meses Abril, Mayo, Junio 
y las variables 3,2,1 corresponden a meses posteriores.

Solo usaremos las variables con información anterior a l mes de Junio.

In [13]:
X = data[[
    'LIMIT_BAL',
    'SEX',
    'EDUCATION',
    'MARRIAGE',
    'AGE',
    'PAY_5',
    'PAY_6',
    'BILL_AMT5',
    'BILL_AMT6',
    'PAY_AMT5',
    'PAY_AMT6',
]]
X

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_5,PAY_6,BILL_AMT5,BILL_AMT6,PAY_AMT5,PAY_AMT6
ID,,,,,,,,,,,
1,20000,2,2,1,24,-2,-2,0,0,0,0
2,120000,2,2,2,26,0,2,3455,3261,0,2000
3,90000,2,2,2,34,0,0,14948,15549,1000,5000
4,50000,2,2,1,37,0,0,28959,29547,1069,1000
5,50000,1,2,1,57,0,0,19146,19131,689,679
...,...,...,...,...,...,...,...,...,...,...,...
29996,220000,1,3,1,39,0,0,31237,15980,5000,1000
29997,150000,1,3,2,43,0,0,5190,0,0,0
29998,30000,1,2,2,37,0,0,20582,19357,2000,3100


In [14]:
y = data[[
    'PAY_AMT4',
    'default payment next month',
]]
y

,PAY_AMT4,default payment next month
ID,,
1,0,1
2,1000,1
3,1000,0
4,1100,0
5,9000,0
...,...,...
29996,3047,0
29997,129,0
29998,4200,1


La variable "default payment next month" es binaria por lo que un modelo de clasificación es ideal

Se implementaron lo siguientes modelos de clasificación.
- LogisticRegression
- DecisionTreeClassifier
- GradientBoostingClassifier
- ExtraTreesClassifier

Las métricas a mostrar son Accuracy, F1, ROC AUC.

Por otro lado la variable "PAY_AMT4" requiere de un modelo de regresión.

Se implementaron los siguientes modelos de regresión.
- LinearRegression
- DecisionTreeRegressor
- GradientBoostingRegressor
- ExtraTreesRegressor

Las métricas a mostrar son $R^2$, RMSE.

In [32]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.model_selection import train_test_split
import sklearn.metrics as metrics

In [1]:
# El proposito de esta función es mostrar las metricas correspondientes a cada modelo
def report(y, y_pred, mode):
    if mode=="classifier":
        accuracy = metrics.accuracy_score(y, y_pred)
        f1_score = metrics.f1_score(y, y_pred)
        roc_auc = metrics.roc_auc_score(y, y_pred)
        print("{} \t {:.4f}".format("accuracy", accuracy))
        print("{} \t {:.4f}".format("f1_score", f1_score))
        print("{} \t {:.4f}".format("roc_auc", roc_auc))
    if mode=="regression":
        rmse = metrics.root_mean_squared_error(y, y_pred) 
        r_squared = metrics.r2_score(y, y_pred)
        print("{} \t {:.4f}".format("r2", r_squared))
        print("{} \t {}".format("rmse", int(round(rmse,0))))        

In [30]:
# Se dividen los datos en datos de entrenamiento  y de prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Modelos de clasificación

In [36]:
logisticRegression = LogisticRegression(max_iter=1000000)
logisticRegression.fit(X_train, y_train["default payment next month"])
report(y_test["default payment next month"], logisticRegression.predict(X_test), "classifier")

accuracy 	 0.7892
f1_score 	 0.1085
roc_auc 	 0.5255


In [37]:
gradientBoostingClassifier = GradientBoostingClassifier()
gradientBoostingClassifier.fit(X_train, y_train["default payment next month"])
report(y_test["default payment next month"], gradientBoostingClassifier.predict(X_test), "classifier")

accuracy 	 0.7965
f1_score 	 0.2946
roc_auc 	 0.5795


In [38]:
extraTreesClassifier = ExtraTreesClassifier()
extraTreesClassifier.fit(X_train, y_train["default payment next month"])
report(y_test["default payment next month"], extraTreesClassifier.predict(X_test), "classifier")

accuracy 	 0.7835
f1_score 	 0.3224
roc_auc 	 0.5861


## Modelos de regresión

In [64]:
linearRegression = LinearRegression()
linearRegression.fit(X_train, y_train["PAY_AMT4"])
report(y_test["PAY_AMT4"], linearRegression.predict(X_test), "regression")

r2 	 0.1456
rmse 	 14756


In [40]:
gradientBoostingRegressor = GradientBoostingRegressor()
gradientBoostingRegressor.fit(X_train, y_train["PAY_AMT4"])
report(y_test["PAY_AMT4"], gradientBoostingRegressor.predict(X_test), "regression")

r2 	 0.5507
rmse 	 10700


In [41]:
extraTreesRegressor = ExtraTreesRegressor()
extraTreesRegressor.fit(X_train, y_train["PAY_AMT4"])
report(y_test["PAY_AMT4"], extraTreesRegressor.predict(X_test), "regression")

r2 	 0.6187
rmse 	 9857


## Redes Neuronales
Adicionalmente se uso un modelo de red neuronal simple, para esto se realizaron  cambios en el conjunto de datos.

Se uso la función get_dummies convertir las variables "EDUCATION","MARRIAGE","AGE","PAY_5","PAY_6" en variables binarias. 

Se escalaron las variables "LIMIT_BAL","BILL_AMT5","BILL_AMT6","PAY_AMT4","PAY_AMT5","PAY_AMT6" dividiendo entre el valor máximo.

In [16]:
X.max()

LIMIT_BAL    1000000
SEX                2
EDUCATION          6
MARRIAGE           3
AGE               79
PAY_5              8
PAY_6              8
BILL_AMT5     927171
BILL_AMT6     961664
PAY_AMT5      426529
PAY_AMT6      528666
dtype: int64

In [19]:
import tensorflow as tf
X_scaled = pd.get_dummies(X, drop_first=True, columns=["EDUCATION","MARRIAGE","AGE","PAY_5","PAY_6"], dtype=int)
y_scaled=y.iloc[:,:]
y_scaled["PAY_AMT4"]=y_scaled["PAY_AMT4"].astype(float)*1e-6
X_scaled[["LIMIT_BAL","BILL_AMT5","BILL_AMT6","PAY_AMT5","PAY_AMT6"]]=X_scaled[["LIMIT_BAL","BILL_AMT5","BILL_AMT6","PAY_AMT5","PAY_AMT6"]].astype(float)*1E-6
Xs_train, Xs_test, ys_train, ys_test = train_test_split(X_scaled, y_scaled, test_size=0.2)

In [20]:
X_scaled

,LIMIT_BAL,SEX,BILL_AMT5,BILL_AMT6,PAY_AMT5,PAY_AMT6,EDUCATION_1,EDUCATION_2,EDUCATION_3,EDUCATION_4,...,PAY_5_8,PAY_6_-1,PAY_6_0,PAY_6_2,PAY_6_3,PAY_6_4,PAY_6_5,PAY_6_6,PAY_6_7,PAY_6_8
ID,,,,,,,,,,,,,,,,,,,,,
1,0.02,2,0.000000,0.000000,0.000000,0.000000,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0.12,2,0.003455,0.003261,0.000000,0.002000,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0
3,0.09,2,0.014948,0.015549,0.001000,0.005000,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0.05,2,0.028959,0.029547,0.001069,0.001000,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0
5,0.05,1,0.019146,0.019131,0.000689,0.000679,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29996,0.22,1,0.031237,0.015980,0.005000,0.001000,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
29997,0.15,1,0.005190,0.000000,0.000000,0.000000,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
29998,0.03,1,0.020582,0.019357,0.002000,0.003100,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0


In [58]:
model = tf.keras.models.Sequential([
  tf.keras.Input(shape=(88,)),  
  tf.keras.layers.Dense(512, activation='relu'),
  tf.keras.layers.Dense(1)
])

model.compile(optimizer='adam',
              loss='mean_squared_error',
              metrics=['accuracy', 'r2_score', 'root_mean_squared_error', 'AUC', 'f1_score'])

In [59]:
model.fit(Xs_train, ys_train["default payment next month"])
score = model.evaluate(Xs_test, ys_test["default payment next month"])
print("{} \t {:.4f}".format("accuracy", score[1]))
print("{} \t {:.4f}".format("roc_auc", score[4]))
print("{} \t {:.4f}".format("f1_score", score[5]))

750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - AUC: 0.6224 - accuracy: 0.7883 - f1_score: 0.3577 - loss: 0.1646 - r2_score: 0.0335 - root_mean_squared_error: 0.4057
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.6841 - accuracy: 0.8029 - f1_score: 0.3551 - loss: 0.1549 - r2_score: 0.0851 - root_mean_squared_error: 0.3935
accuracy 	 0.7973
roc_auc 	 0.6844
f1_score 	 0.3665


In [60]:
model.fit(Xs_train, ys_train["PAY_AMT4"])
score = model.evaluate(Xs_test, ys_test["PAY_AMT4"])
print("{} \t {:.4f}".format("accuracy", score[1]))
print("{} \t \t {:.4f}".format("r2", score[2]))
print("{} \t \t {:.4f}".format("rmse", score[3]))

750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - AUC: 0.4930 - accuracy: 0.2158 - f1_score: 0.0097 - loss: 0.0052 - r2_score: -32.5105 - root_mean_squared_error: 0.0646
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.5325 - accuracy: 0.2143 - f1_score: 0.0091 - loss: 2.9569e-04 - r2_score: -0.5845 - root_mean_squared_error: 0.0171
accuracy 	 0.2100
r2 	 	 -0.1758
rmse 	 	 0.0189


<h2 style="text-align: center;">Modelos de Clasificación</h2>

|Modelo|Accuracy|F1 score|ROC AUC|
|------|--------|--|-------|
|LogisticRegression|0.7892|0.1085|0.5255|
|DecisionTreeClassifier|0.7965|0.2946|0.5795|
|ExtraTreesClassifier|0.7835|0.3224|0.5861|
|Neural Network|0.7973|0.3665|0.6844|

<h2 style="text-align: center;">Modelos de regresión</h2>

|Modelo|$R^2$|RMSE|
|------|-----|----|
|LinearRegression|0.1456|14756|
|GradientBoostingRegressor|0.5507|10700|
|ExtraTreesRegressor|0.6187|9857
|Neural Network|-0.1758|0.0189|

## Conclusiones
La variable 'default payment next month' se logro modelar con una valor de Accuracy cercano a 0.8 en los tres modelos de clasificación y  la red neuronal. El modelo con mejor rendimeinto fue la red neuronal obteniendo el puntaje máximo en las metricas sleccionadas.

En la practica el modelo acierta en predecir si el cliente pagara su deuda crediticia el mes próximo 4 de cada 5 casos. Este nivel de precisión es muy bajo y de implementarse puede causar una gran número de falsos positivos. Implementar variables externas como la inflación, festividades y ciclos empresariales podría aumentar la precisión de las predicciones así como suministrar datos adicionales de los clientes.

La variable 'PAY_AMT4' obtuvo un rendimiento muy inferior, el mejor modelo en las pruebas fue ExtraTreesRegressor con un valor de $R^2$=0.6187 y RMSE=9857. En este caso el nivel de precisión del mejor modelo no es confiable para predecir pagos individuales pero podria dar un aproximado  del ingreso total de los pagos recibidos para el mes próximo.

In [84]:
# suma de los pagos en los datos de prueba
real = ys_test["PAY_AMT4"].sum()*1e6
# suma de las predicciones en los datos de prueba usando el modelo de red neuronal
predict = model.predict(Xs_test).sum()*1e6
print(real, predict)
print("{:.4f}".format(100*abs(real-predict)/real))

188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 611us/step
29813562.0 29474424.0
1.1375


El modelo neuronal puede predecir el total de pagos realizados el mes de Junio con un error porcentual menor a 1.2%.

1.	De las deficiencias en los datos, ¿cuáles y como las identificaste?
  
    La base de datos es consta de  25,000 observaciones y 23 variables, si se quiere mejorar la calidad de la base de datos se podría incrementar el número de variables e implementar criterios para identificar las más relevantes. Ampliar la base de datos o seleccionar una muestra representativa para tener datos bien distribuidos también puede mejorar muchos el rendimiento de los modelos.
  
3.	De realizar creación de variables, explica cuales hiciste y por qué.
  
    Realice creación de variables para aplicar usar los modelos de redes neuronales, pase loas variables de clasificación a vectores de variables binarias usando lo que se llama one hot encoding.

5.	De los modelos realizados, ¿cómo seleccionaste al mejor?

    Las métricas dan una buena idea del rendimiento de cada modelo con los datos de prueba.  Accuracy, F1, ROC AUC para los clasificadores y $R^2$, RMSE para los modelos de regresión.

7.	¿Qué desafíos encontraste y cómo los superaste?

    El mayor desafío fue encontrar un modelo funcional para la variable 'PAY_AMT4', finalmente encontré que el modelo de red neuronal es mala para predecir casos individuales en este ejercicio, pero es excelente realizando una predicción general.


1.	Establece con tus propias palabras, algunas buenas prácticas y funciones recomendadas para optimizar operaciones de lectura, escritura y manipulación en Spark/PySpark.
   Hay operaciones que son mas costosas de realizar en  sistemas paralelos, como los ordenamientos por eso es recomendable evitar ordenamientos. Otra buena practica es usar las funciones incluidas en pyspark como pyspark.pandas en vez de funciones excteranas ya que están optimizadas para este caso especial de uso.
Siempre se debe evitar cargar los cálculos en una partición única. La fortaleza de el computo distribuido es que puede ser mas rápido al repartir el trabajo entre muchos ordenadores. 
  
3.	Indica las pruebas estadísticas que has utilizado como parte del desarrollo de una solución de ciencia de datos.
    Una función que he usado es la función de correlación, esta indica la relacion entre variables y es un criterio muy útil para encontrar dependencias o descartar variables.  Otro concepto importante es el de distribución, en los métodos de Monte Carlo se usan distribuciones de probabilidad  para seleccionar las configuraciones que contribuyen al cambio en el estado de un sistema estadístico.

5.	En el contexto de Machine Learning y Ciencia de datos, explica:

    a.	No Free Lunch Theorem

    No hay almuerzo gratis, es una forma de explicar como un modelo sobre-ajustado conlleva como precio que usando un conjunto diferente de datos a los datos de entrenamiento el modelo no funcionara bien ya que esta optimizado para el conjunto de datos de entrenamiento.


    b.	Occam’s Razor

    La navaja de Occam explica que la explicación más simple es por lo general la mas probable, en machine learning el modelo sencillo que describe la relación entre un conjunto de datos es el mejor modelo.

    c.	Data Leakage

    En física hay una frase llamada, usar lo que quieres demostrar, sin darte cuenta usas un argumento que no deberías poder usar. En este caso usar datos directamente relacionados una variable que se busca predecir tiene el mismo efecto. En un caso de aplicación real el modelo no servirá ya que no se puede hacer al misma trampa y usar datos inexistentes.


## Referencias
Yeh, I-Cheng. "Default of Credit Card Clients." UCI Machine Learning Repository, 2009, https://doi.org/10.24432/C55S3H.